In [2]:
from pathlib import Path

import torch 
from torch import nn

from torchvision import datasets,transforms
from torchvision.models import resnet18,ResNet18_Weights
from torch.utils.data import DataLoader,Subset,random_split

In [3]:
data_path = Path('data/')


In [4]:
list(data_path.iterdir())

[WindowsPath('data/Tomato_Bacterial_spot'),
 WindowsPath('data/Tomato_Early_blight'),
 WindowsPath('data/Tomato_healthy'),
 WindowsPath('data/Tomato_Late_blight'),
 WindowsPath('data/Tomato_Leaf_Mold'),
 WindowsPath('data/Tomato_Septoria_leaf_spot'),
 WindowsPath('data/Tomato_Spider_mites_Two_spotted_spider_mite'),
 WindowsPath('data/Tomato__Target_Spot'),
 WindowsPath('data/Tomato__Tomato_mosaic_virus'),
 WindowsPath('data/Tomato__Tomato_YellowLeaf__Curl_Virus')]

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [6]:
plant_dataset = datasets.ImageFolder(root=data_path)
plant_dataset.class_to_idx

{'Tomato_Bacterial_spot': 0,
 'Tomato_Early_blight': 1,
 'Tomato_Late_blight': 2,
 'Tomato_Leaf_Mold': 3,
 'Tomato_Septoria_leaf_spot': 4,
 'Tomato_Spider_mites_Two_spotted_spider_mite': 5,
 'Tomato__Target_Spot': 6,
 'Tomato__Tomato_YellowLeaf__Curl_Virus': 7,
 'Tomato__Tomato_mosaic_virus': 8,
 'Tomato_healthy': 9}

In [7]:
plant_dataset.classes

['Tomato_Bacterial_spot',
 'Tomato_Early_blight',
 'Tomato_Late_blight',
 'Tomato_Leaf_Mold',
 'Tomato_Septoria_leaf_spot',
 'Tomato_Spider_mites_Two_spotted_spider_mite',
 'Tomato__Target_Spot',
 'Tomato__Tomato_YellowLeaf__Curl_Virus',
 'Tomato__Tomato_mosaic_virus',
 'Tomato_healthy']

In [22]:
torch.manual_seed(42)

train_size = int(0.7*len(plant_dataset))
val_size = int(0.15*len(plant_dataset))
test_size = len(plant_dataset) - train_size - val_size

train_subset,val_subset,test_subset = random_split(
    plant_dataset,
    [train_size,val_size,test_size]
)

In [23]:
train_dataset = datasets.ImageFolder(root=data_path,
                                     transform=train_transform)

val_dataset = datasets.ImageFolder(root=data_path,
                                   transform=test_transform)

test_dataset = datasets.ImageFolder(root=data_path,
                                    transform=test_transform)

In [24]:
train_dataset = Subset(train_dataset,train_subset.indices)
val_dataset = Subset(val_dataset,val_subset.indices)
test_dataset = Subset(test_dataset,test_subset.indices)

In [25]:
train_loader = DataLoader(train_dataset,
                          batch_size=32,
                          shuffle=True)

val_loader = DataLoader(val_dataset,
                        batch_size=32,
                        shuffle=False)

test_loader = DataLoader(test_dataset,
                         batch_size=32,
                         shuffle=False)

In [26]:
weights = ResNet18_Weights.DEFAULT
resnet_model = resnet18(weights=weights)
resnet_model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [32]:
for param in resnet_model.parameters():
    param.requires_grad = False

In [33]:
for param in resnet_model.layer4.parameters():
    param.requires_grad = True

In [27]:
torch.manual_seed(42)

resnet_model.fc = nn.Linear(in_features=resnet_model.fc.in_features,
                            out_features=len(plant_dataset.classes))

resnet_model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [34]:
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, resnet_model.parameters()),
    lr=1e-4
)

In [35]:
def train_step(model,dataloader,loss_fn,optimizer):
    model.train()

    train_loss = 0
    correct = 0
    total = 0

    for images,labels in dataloader:

        outputs = model(images)
        loss = loss_fn(outputs,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        preds = outputs.argmax(dim=1)
        correct += (preds==labels).sum().item()
        total += labels.size(0)

    train_loss /= len(dataloader)
    train_acc = (correct/total)*100

    return train_loss,train_acc

In [36]:
def val_step(model,dataloader,loss_fn):
    model.eval()

    val_loss = 0
    correct = 0
    total = 0

    with torch.inference_mode():

        for images,labels in dataloader:
            outputs = model(images)
            loss = loss_fn(outputs,labels)

            val_loss += loss.item()

            preds = outputs.argmax(dim=1)
            correct += (preds==labels).sum().item()
            total += labels.size(0)

    val_loss /= len(dataloader)
    val_acc = (correct/total)*100

    return val_loss,val_acc

In [37]:
epochs = 5

for epoch in range(epochs):

    train_loss,train_acc = train_step(resnet_model,
                                      train_loader,
                                      loss_fn,
                                      optimizer)

    val_loss,val_acc = val_step(resnet_model,
                                val_loader,
                                loss_fn)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

Epoch 1/5 | Train Loss: 0.0745 | Train Acc: 97.7514 | Val Loss: 0.0548 | Val Acc: 98.1258
Epoch 2/5 | Train Loss: 0.0424 | Train Acc: 98.6437 | Val Loss: 0.0504 | Val Acc: 98.5006
Epoch 3/5 | Train Loss: 0.0322 | Train Acc: 99.0274 | Val Loss: 0.0587 | Val Acc: 98.1258
Epoch 4/5 | Train Loss: 0.0277 | Train Acc: 99.1166 | Val Loss: 0.0405 | Val Acc: 98.8338
Epoch 5/5 | Train Loss: 0.0154 | Train Acc: 99.5628 | Val Loss: 0.0423 | Val Acc: 98.6672


In [38]:
def test_step(model,dataloader,loss_fn):
    model.eval()

    test_loss = 0
    correct = 0
    total = 0

    with torch.inference_mode():
        for images,labels in dataloader:
            outputs = model(images)
            loss = loss_fn(outputs,labels)

            test_loss += loss.item()

            preds = outputs.argmax(dim=1)
            correct += (preds==labels).sum().item()
            total += labels.size(0)

    test_loss /= len(dataloader)
    test_acc = (correct/total)*100


    return test_loss,test_acc


test_loss,test_acc = test_step(resnet_model,
                               test_loader,
                               loss_fn)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

Test Loss: 0.0496
Test Accuracy: 98.3770


In [39]:
torch.save(resnet_model.state_dict(),'PDC_ResNet18.pth')